# Fase 4.2 — Comparación preespecificada, calibración y selección del modelo

**Tesis v2 · Eider Johan Díaz Berrío · MCDA EAFIT**

Con las predicciones fuera de pliegue del notebook 09, calculo las métricas del protocolo: *log loss*, Brier escalado, AUC, PR-AUC, KS, calibración (intercepto, pendiente, ICI) y beneficio neto. Todas llevan IC 95 % por *bootstrap* de clientes, con 2.000 réplicas.

Después aplico **la regla de selección tal como quedó escrita antes de ver resultados** (sección 8 del protocolo y sección 5 de la enmienda v1.1). La regla tiene cuatro pasos:

1. Descarto los modelos con pendiente de calibración fuera de [0,6; 1,4].
2. Entre los restantes, tomo el de menor *log loss*.
3. Si un modelo más simple tiene una diferencia de *log loss* cuyo IC pareado incluye 0, prefiero el más simple. El orden de complejidad es M1 < M2/M2F < M3 < M6 < M5.
4. Comparo el modelo elegido con la tasa base (M0a) y declaro si la mejora en probabilidad es demostrable.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.metrics import roc_auc_score
from statsmodels.nonparametric.smoothers_lowess import lowess

from tesis_v2 import config, contexto, diseno, figuras, io_datos, metricas as mt, modelos

warnings.filterwarnings("ignore")
pd.set_option("display.width", 180, "display.max_columns", 40, "display.max_colwidth", 60)
figuras.estilo()

z = np.load(config.PRIVADO / "oof_principal.npz")
y, grupos = z["y"], z["grupo"]
CLAVES = ["M0a", "M0b", "M1", "M2", "M2F", "M3", "M4", "M5", "M6", "M7_60", "M7_30"]
Ps = {k: z[k] for k in CLAVES}
NOMBRES = {k: modelos.bateria()[k]().nombre for k in CLAVES}
print(f"{len(y)} créditos, {int(y.sum())} eventos, {len(np.unique(grupos))} clientes; {Ps['M2'].shape[0]} repeticiones")

733 créditos, 23 eventos, 357 clientes; 20 repeticiones


## 1. Tabla única de resultados

Cada cifra es la media de las 20 repeticiones de la métrica calculada sobre los 733 créditos fuera de pliegue. El IC sale del *bootstrap* por cliente, que remuestrea clientes y no créditos: los créditos de una misma persona no son independientes.

In [2]:
boot = mt.bootstrap(Ps, y, grupos, B=2000)
tabla = mt.tabla_resultados(Ps, y, boot, NOMBRES)
io_datos.guardar_resultado(tabla.round(5), "fase4", "tabla_unica_resultados")
cols = ["modelo", "nombre", "log_loss", "log_loss_ic", "brier_escalado", "brier_escalado_ic", "auc", "auc_ic",
        "pr_auc", "ks", "intercepto_cal", "pendiente_cal", "pendiente_cal_ic", "ici", "pd_media"]
tabla[cols].round(4)

,modelo,nombre,log_loss,log_loss_ic,brier_escalado,brier_escalado_ic,auc,auc_ic,pr_auc,ks,intercepto_cal,pendiente_cal,pendiente_cal_ic,ici,pd_media
0,M0a,M0a Tasa base,0.1396,0.097–0.185,-0.0002,-0.010–-0.000,0.4739,0.451–0.496,0.0300,0.1570,-0.0258,-4.2387,-9.400–-0.991,0.0038,0.0322
1,M0b,M0b Regla de políticas,0.1416,0.099–0.187,-0.0039,-0.015–-0.002,0.3908,0.319–0.448,0.0253,0.2102,-0.0541,-2.5504,-5.169–-0.146,0.0114,0.0331
2,M1,M1 Scorecard WoE,0.1511,0.103–0.204,-0.0172,-0.048–0.002,0.5471,0.434–0.657,0.0434,0.2080,0.0076,0.0768,-0.337–0.570,0.0172,0.0312
3,M2,M2 Logística penalizada,0.1387,0.095–0.186,0.0046,-0.015–0.018,0.6294,0.502–0.747,0.0649,0.3062,-0.0042,0.6181,-0.423–1.652,0.0083,0.0315
4,M2F,M2F Logística de Firth,0.1483,0.105–0.195,-0.0482,-0.136–0.000,0.5996,0.482–0.719,0.0487,0.2689,-0.3115,0.2215,-0.263–0.769,0.0175,0.0418
5,M3,M3 Bayesiana con prior Home Credit,0.1366,0.094–0.183,0.0104,-0.013–0.029,0.6531,0.525–0.781,0.0760,0.3617,-0.0052,0.7720,-0.043–1.770,0.0071,0.0315
6,M4,M4 Apilado v1 (HC 80 %),0.1377,0.094–0.185,0.0194,-0.028–0.064,0.6442,0.518–0.776,0.1139,0.3429,-0.0184,0.5746,-0.012–1.253,0.0098,0.0319
7,M5,M5 Boosting monótono,0.1371,0.094–0.183,0.0051,-0.022–0.023,0.6333,0.515–0.754,0.0667,0.3132,-0.0020,0.7242,0.074–1.359,0.0090,0.0314
8,M6,M6 Supervivencia (tiempo discreto),0.1426,0.105–0.182,-0.0115,-0.058–0.006,0.6025,0.481–0.725,0.0705,0.2506,-0.4668,1.1691,-0.501–2.945,0.0193,0.0490
9,M7_60,M7 Auxiliar y_60_12m,0.1389,0.095–0.186,0.0045,-0.028–0.028,0.6419,0.524–0.760,0.0670,0.2995,0.0002,0.5254,-0.098–1.226,0.0079,0.0314


**Cómo leer la tabla.**

- **Referencias.** La tasa base observada es 3,14 %, y el *log loss* de un modelo que siempre predice esa tasa es 0,139. Cualquier modelo útil debe quedar por debajo.
- **Brier escalado.** Mide la fracción del error cuadrático de la tasa base que el modelo elimina. Un valor de 0,01 significa un 1 %.
- **AUC de M0a.** M0a no debería discriminar, pero su AUC queda algo por debajo de 0,5. Es un artefacto conocido de la validación cruzada con pocos eventos: un pliegue de prueba con más eventos deja un entrenamiento con menos, y por tanto una tasa predicha más baja, justo donde hay más eventos.
- **Pendiente de calibración.** Por debajo de 1 indica predicciones demasiado extremas (sobreajuste); por encima de 1, demasiado tímidas.

## 2. Comparaciones pareadas

Como todos los modelos se evaluaron con las mismas particiones y las mismas réplicas *bootstrap*, la diferencia entre dos modelos se estima de forma pareada, mucho más precisa que comparar dos IC por separado. Comparo cada modelo con:

- **M0a (tasa base)**, que responde si el modelo estima mejor la probabilidad que no usar modelo;
- **M1** y **M2**, las referencias del protocolo, con corrección de Holm dentro de cada familia.

In [3]:
comp = []
for ref in ["M0a", "M1", "M2"]:
    for k in CLAVES:
        if k == ref:
            continue
        for met in ["log_loss", "auc"]:
            c = mt.comparacion_pareada(boot, tabla, k, ref, met)
            c["familia"] = f"frente a {ref}"
            comp.append(c)
comp = pd.DataFrame(comp)
for (fam, met), g in comp.groupby(["familia", "metrica"]):
    comp.loc[g.index, "p_holm"] = mt.holm(g["p_bootstrap"])
io_datos.guardar_resultado(comp.round(5), "fase4", "comparaciones_pareadas")
vista = comp[comp["metrica"] == "log_loss"].assign(
    ic95=lambda t: [f"{a:+.4f} a {b:+.4f}" for a, b in zip(t["ic_inf"], t["ic_sup"])])
vista.pivot_table(index="modelo", columns="familia", values="delta", aggfunc="first").round(4).join(
    vista[vista["familia"] == "frente a M0a"].set_index("modelo")[["ic95", "p_holm"]].rename(
        columns={"ic95": "IC Δ frente a M0a", "p_holm": "p Holm frente a M0a"})).loc[[k for k in CLAVES if k != "M0a"]]

,frente a M0a,frente a M1,frente a M2,IC Δ frente a M0a,p Holm frente a M0a
modelo,,,,,
M0b,0.0020,-0.0096,0.0029,+0.0004 a +0.0040,0.000
M1,0.0115,NaN,0.0125,-0.0013 a +0.0276,0.819
M2,-0.0009,-0.0125,NaN,-0.0084 a +0.0074,1.000
M2F,0.0087,-0.0028,0.0096,-0.0055 a +0.0244,1.000
M3,-0.0030,-0.0145,-0.0021,-0.0123 a +0.0067,1.000
M4,-0.0019,-0.0135,-0.0010,-0.0164 a +0.0122,1.000
M5,-0.0025,-0.0141,-0.0016,-0.0109 a +0.0053,1.000
M6,0.0030,-0.0085,0.0039,-0.0046 a +0.0097,1.000
M7_60,-0.0007,-0.0122,0.0002,-0.0125 a +0.0122,1.000


In [4]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), sharey=True)
orden = [k for k in CLAVES if k != "M0a"][::-1]
d0 = comp[(comp["familia"] == "frente a M0a") & (comp["metrica"] == "log_loss")].set_index("modelo").loc[orden]
axes[0].errorbar(1000 * d0["delta"], range(len(orden)), xerr=[1000 * (d0["delta"] - d0["ic_inf"]), 1000 * (d0["ic_sup"] - d0["delta"])],
                 fmt="o", color=figuras.AZUL, capsize=2)
axes[0].axvline(0, color=figuras.GRIS, lw=1)
axes[0].set_xlabel("Δ log loss frente a la tasa base (× 1.000; negativo = mejor)")
axes[0].set_yticks(range(len(orden)), [f"{k}  {NOMBRES[k].split(' ', 1)[1]}" for k in orden], fontsize=7.5)
t = tabla.set_index("modelo").loc[orden]
ic = np.array([list(map(float, s.split("–"))) for s in t["auc_ic"]])
axes[1].errorbar(t["auc"], range(len(orden)), xerr=[t["auc"] - ic[:, 0], ic[:, 1] - t["auc"]], fmt="o", color=figuras.NARANJA, capsize=2)
axes[1].axvline(0.5, color=figuras.GRIS, lw=1)
axes[1].set_xlabel("AUC (IC 95 % bootstrap por cliente)")
fig.suptitle("Desempeño fuera de pliegue — validación cruzada 5 × 20 agrupada por cliente", fontweight="bold", fontsize=10)
fig.tight_layout()
figuras.guardar(fig, "fase4", "M01", "comparacion_modelos",
                "Ningún modelo mejora de forma demostrable a la tasa base en log loss; los mejores discriminan (AUC 0,63–0,65, límite inferior ≥ 0,50) y la regla de políticas va al revés.")

## 3. Calibración

La calibración es la propiedad más importante para el uso previsto: estimar una probabilidad para fijar precio, provisión o banda. Grafico la curva *lowess* de lo observado frente a lo predicho. Uso la predicción fuera de pliegue promediada entre las 20 repeticiones; la banda es un *bootstrap* por cliente de 200 réplicas. El eje se limita a 0–20 %, donde está casi toda la cartera.

In [5]:
def curva(p, yy, rejilla):
    s = lowess(yy, p, frac=0.75, it=0, return_sorted=True)
    return np.interp(rejilla, s[:, 0], s[:, 1], left=np.nan, right=np.nan)


rejilla = np.linspace(0.005, 0.15, 60)
MOSTRAR = ["M2", "M3", "M4", "M5", "M6", "M1"]
fig, axes = plt.subplots(2, 3, figsize=(11, 6.6), sharex=True, sharey=True)
idx_boot = list(mt.indices_bootstrap(grupos, B=200, semilla=config.SEMILLA + 1))
for ax, k in zip(axes.ravel(), MOSTRAR):
    pbar = Ps[k].mean(axis=0)
    c = curva(pbar, y, rejilla)
    banda = np.array([curva(pbar[i], y[i], rejilla) for i in idx_boot])
    lo, hi = np.nanpercentile(banda, [2.5, 97.5], axis=0)
    ax.fill_between(rejilla, lo, hi, color=figuras.AZUL, alpha=0.18, lw=0)
    ax.plot(rejilla, c, color=figuras.AZUL, lw=1.8)
    ax.plot([0, 0.15], [0, 0.15], color=figuras.GRIS, ls="--", lw=1)
    alto = np.quantile(pbar, 0.99)
    ax.axvspan(alto, 0.15, color="#f2f2f2", zorder=0)
    fila = tabla.set_index("modelo").loc[k]
    ax.set_title(f"{k}: pendiente {fila['pendiente_cal']:.2f}, ICI {fila['ici']:.3f}", fontsize=8.5)
    ax.set_xlim(0, 0.15)
    ax.set_ylim(0, 0.2)
for ax in axes[-1]:
    ax.set_xlabel("PD predicha")
for ax in axes[:, 0]:
    ax.set_ylabel("Tasa observada (lowess)")
fig.suptitle("Curvas de calibración fuera de pliegue (zona gris: por encima del percentil 99 de la PD)", fontweight="bold", fontsize=10)
fig.tight_layout()
figuras.guardar(fig, "fase4", "M02", "calibracion",
                "Las bandas de calibración son muy anchas por encima del 5 % de PD: con 23 eventos solo se puede verificar la calibración en el rango bajo.")

**Lectura de la calibración.**

- **Rango central (PD de 3 a 8 %).** Las curvas de M2, M3 y M4 siguen la diagonal dentro de la banda. M3 es la más cercana a la diagonal en todo el rango (ICI 0,007).
- **Cola baja (PD < 2 %).** M2 subestima: la tasa observada ronda el 3 % a 6 %. Son pocos créditos, y la banda incluye la diagonal en el borde.
- **Por encima del 8 %.** Cada curva descansa en un puñado de eventos, la banda se abre y no permite afirmar nada.
- **M5 y M1.** Muestran curvas no monótonas: la tasa observada cae para las PD más altas. Es la huella del sobreajuste, porque las predicciones más extremas son las menos confiables.

Hay dos desviaciones sistemáticas:

- **M6 sobrestima en promedio** (PD media 4,9 % frente a 3,1 % observado). No es un error del modelo sino una diferencia de definición. La supervivencia estima la incidencia a 12 meses contando los cierres tempranos como censura, mientras que la etiqueta binaria los cuenta como no evento. Es la misma brecha que vimos en la Fase 1 entre Kaplan-Meier (5,7 %) y la tasa binaria (2,9 %).
- **M1, M2F y M4 tienen pendientes muy por debajo de 1.** Sus predicciones son demasiado extremas: el *scorecard* y Firth sin contracción adicional sobreajustan con 23 eventos, y el apilado v1 hereda pendientes de otra población.

## 4. Regla de selección preespecificada

In [6]:
COMPLEJIDAD = ["M1", "M2", "M2F", "M3", "M6", "M5"]
T = tabla.set_index("modelo")
pasos = []
elegibles = [k for k in COMPLEJIDAD if 0.6 <= T.loc[k, "pendiente_cal"] <= 1.4]
pasos.append(("1. Pendiente de calibración en [0,6; 1,4]", ", ".join(elegibles),
              "; ".join(f"{k}: {T.loc[k, 'pendiente_cal']:.2f}" for k in COMPLEJIDAD if k not in elegibles) or "—"))
mejor = min(elegibles, key=lambda k: T.loc[k, "log_loss"])
pasos.append(("2. Menor log loss entre los elegibles", mejor, f"log loss {T.loc[mejor, 'log_loss']:.4f}"))
elegido, detalle = mejor, []
for k in COMPLEJIDAD:
    if k == mejor or COMPLEJIDAD.index(k) >= COMPLEJIDAD.index(mejor) or k not in elegibles:
        continue
    c = mt.comparacion_pareada(boot, tabla, mejor, k, "log_loss")
    incluye = c["ic_inf"] <= 0 <= c["ic_sup"]
    detalle.append(f"{mejor} − {k}: {c['delta']:+.4f} [{c['ic_inf']:+.4f}; {c['ic_sup']:+.4f}] → {'incluye 0' if incluye else 'no incluye 0'}")
    if incluye:
        elegido = k
        break
pasos.append(("3. Preferir el más simple si el IC pareado incluye 0", elegido, " | ".join(detalle) or "no hay elegibles más simples"))
c0 = mt.comparacion_pareada(boot, tabla, elegido, "M0a", "log_loss")
demuestra = c0["ic_sup"] < 0
pasos.append(("4. Frente a la tasa base (M0a)", "mejora demostrable" if demuestra else "mejora NO demostrable",
              f"Δ log loss {c0['delta']:+.4f} [{c0['ic_inf']:+.4f}; {c0['ic_sup']:+.4f}]"))
seleccion = pd.DataFrame(pasos, columns=["paso", "resultado", "detalle"])
io_datos.guardar_resultado(seleccion, "fase4", "regla_de_seleccion")
(config.RESULTADOS / "fase4" / "modelo_elegido.json").write_text(json.dumps({"modelo": elegido, "nombre": NOMBRES[elegido],
    "mejor_log_loss": mejor, "mejora_frente_tasa_base_demostrable": bool(demuestra)}, ensure_ascii=False, indent=1))
seleccion

,paso,resultado,detalle
0,"1. Pendiente de calibración en [0,6; 1,4]","M2, M3, M6, M5",M1: 0.08; M2F: 0.22
1,2. Menor log loss entre los elegibles,M3,log loss 0.1366
2,3. Preferir el más simple si el IC pareado incluye 0,M2,M3 − M2: -0.0021 [-0.0049; +0.0006] → incluye 0
3,4. Frente a la tasa base (M0a),mejora NO demostrable,Δ log loss -0.0009 [-0.0084; +0.0074]


**Lectura de la regla.**

- **Paso 1.** Quedan descartados por calibración M1 (pendiente 0,08) y M2F (0,22): sobreajustan.
- **Paso 2.** El menor *log loss* lo tiene **M3**, la logística con el *prior* de Home Credit.
- **Paso 3.** M3 mejora a M2 en solo 0,002 de *log loss*, y el IC pareado incluye el 0 (−0,005 a +0,001). La regla escrita antes de ver resultados ordena preferir el modelo más simple, así que el **modelo elegido es M2, la logística penalizada solo con datos de FINTECH**.
- **Paso 4.** Frente a la tasa base, la mejora de M2 en *log loss* **no es demostrable**: Δ −0,0009, con IC −0,008 a +0,007. Ningún otro modelo la demuestra tampoco.

**Cómo lo leo.** El resultado más honesto de la tesis es este: con 23 eventos, **ningún modelo estima la probabilidad de forma demostrablemente mejor que la tasa base**. La discriminación, en cambio, sí es real: los mejores AUC tienen su límite inferior en 0,50 o por encima. Las dos cosas son compatibles. Un modelo puede ordenar mejor que el azar y, a la vez, sus probabilidades individuales pueden aportar tan poca información adicional que el *log loss* no lo distingue de predecir 3,1 % para todos.

**Sobre M3.** Es el candidato natural para la Fase 6, si la operación acumula eventos. Tiene el menor *log loss*, la mejor calibración (pendiente 0,77) y el AUC puntual más alto (0,65). Su ventaja no es demostrable hoy, y la regla no permite elegirlo; lo dejo registrado como hipótesis a confirmar con datos nuevos, no como conclusión.

**Sobre la regla de políticas (M0b).** Es significativamente *peor* que la tasa base: AUC 0,39 y Δ *log loss* +0,002 con IC que excluye el 0. El *proxy* marca sobre todo a recurrentes con mora previa de 30 días o más, y los recurrentes tienen menos riesgo que los nuevos. Es decir: las reglas evaluables, aplicadas a créditos que ya pasaron la política, señalan al grupo equivocado.

## 5. Corrección del optimismo de Harrell para el modelo elegido

La validación cruzada ya es una estimación fuera de muestra. El *bootstrap* de Harrell es la segunda vía que pide el protocolo y sirve de contraste. Funciona así:

1. Ajusto el modelo elegido, con su selección interna de hiperparámetros, en una muestra *bootstrap* de clientes.
2. Mido cuánto mejor le va en esa muestra que en la muestra original. Esa diferencia es el optimismo.
3. Promedio el optimismo en 500 réplicas y lo resto del desempeño aparente, que es el del modelo ajustado y evaluado en todos los datos.

In [7]:
D, ctx, _ = contexto.cargar()
assert (D["y"].to_numpy() == y).all()
crear = modelos.bateria()[elegido]


def medir(p, yy):
    r = mt._rapidas(p.reshape(-1, 1), yy.astype(float))
    return np.array([r["log_loss"][0], r["auc"][0], r["pendiente_cal"][0], r["brier_escalado"][0]])


aparente = medir(crear().ajustar(D, ctx).predecir(D), y)


def replica(idx):
    Db = D.iloc[idx].reset_index(drop=True)
    m = crear().ajustar(Db, ctx)
    return medir(m.predecir(Db), Db["y"].to_numpy()), medir(m.predecir(D), y)


idxs = list(mt.indices_bootstrap(grupos, B=500, semilla=config.SEMILLA + 2))
res = Parallel(n_jobs=10)(delayed(replica)(i) for i in idxs)
entreno, prueba = np.array([r[0] for r in res]), np.array([r[1] for r in res])
opt = entreno - prueba
corregida = aparente - np.nanmean(opt, axis=0)
# Pendiente: con un modelo tan penalizado, la pendiente aparente de la réplica es inestable (se dispara cuando la
# réplica encoge casi hasta una constante). Uso la variante de Harrell que reporta la mediana de la pendiente del
# modelo de cada réplica evaluado en la muestra original.
print(f"Réplicas con pendiente aparente no definida: {int(np.isnan(entreno[:, 2]).sum())} de {len(res)}")
corregida[2] = np.nanmedian(prueba[:, 2])
harrell = pd.DataFrame({"metrica": ["log_loss", "auc", "pendiente_cal", "brier_escalado"], "aparente": aparente,
                        "optimismo_medio": np.r_[np.nanmean(opt[:, :2], axis=0), np.nan, np.nanmean(opt[:, 3])],
                        "corregida": corregida,
                        "validacion_cruzada_5x20": [T.loc[elegido, m] for m in ["log_loss", "auc", "pendiente_cal", "brier_escalado"]]})
io_datos.guardar_resultado(harrell.round(4), "fase4", "optimismo_harrell")
harrell.round(4)

src/tesis_v2/metricas.py:61: RuntimeWarning: All-NaN slice encountered
  if np.nanmax(np.abs(np.r_[da, db])) < 1e-8:


src/tesis_v2/metricas.py:61: RuntimeWarning: All-NaN slice encountered
  if np.nanmax(np.abs(np.r_[da, db])) < 1e-8:
src/tesis_v2/metricas.py:61: RuntimeWarning: All-NaN slice encountered
  if np.nanmax(np.abs(np.r_[da, db])) < 1e-8:
src/tesis_v2/metricas.py:61: RuntimeWarning: All-NaN slice encountered
  if np.nanmax(np.abs(np.r_[da, db])) < 1e-8:


src/tesis_v2/metricas.py:61: RuntimeWarning: All-NaN slice encountered
  if np.nanmax(np.abs(np.r_[da, db])) < 1e-8:
src/tesis_v2/metricas.py:61: RuntimeWarning: All-NaN slice encountered
  if np.nanmax(np.abs(np.r_[da, db])) < 1e-8:
src/tesis_v2/metricas.py:61: RuntimeWarning: All-NaN slice encountered
  if np.nanmax(np.abs(np.r_[da, db])) < 1e-8:
src/tesis_v2/metricas.py:61: RuntimeWarning: All-NaN slice encountered
  if np.nanmax(np.abs(np.r_[da, db])) < 1e-8:


src/tesis_v2/metricas.py:61: RuntimeWarning: All-NaN slice encountered
  if np.nanmax(np.abs(np.r_[da, db])) < 1e-8:
src/tesis_v2/metricas.py:61: RuntimeWarning: All-NaN slice encountered
  if np.nanmax(np.abs(np.r_[da, db])) < 1e-8:


src/tesis_v2/metricas.py:61: RuntimeWarning: All-NaN slice encountered
  if np.nanmax(np.abs(np.r_[da, db])) < 1e-8:
src/tesis_v2/metricas.py:61: RuntimeWarning: All-NaN slice encountered
  if np.nanmax(np.abs(np.r_[da, db])) < 1e-8:


src/tesis_v2/metricas.py:61: RuntimeWarning: All-NaN slice encountered
  if np.nanmax(np.abs(np.r_[da, db])) < 1e-8:
src/tesis_v2/metricas.py:61: RuntimeWarning: All-NaN slice encountered
  if np.nanmax(np.abs(np.r_[da, db])) < 1e-8:
src/tesis_v2/metricas.py:61: RuntimeWarning: All-NaN slice encountered
  if np.nanmax(np.abs(np.r_[da, db])) < 1e-8:
src/tesis_v2/metricas.py:61: RuntimeWarning: All-NaN slice encountered
  if np.nanmax(np.abs(np.r_[da, db])) < 1e-8:


... (45 outputs left out)

src/tesis_v2/metricas.py:61: RuntimeWarning: All-NaN slice encountered
  if np.nanmax(np.abs(np.r_[da, db])) < 1e-8:
src/tesis_v2/metricas.py:61: RuntimeWarning: All-NaN slice encountered
  if np.nanmax(np.abs(np.r_[da, db])) < 1e-8:


src/tesis_v2/metricas.py:61: RuntimeWarning: All-NaN slice encountered
  if np.nanmax(np.abs(np.r_[da, db])) < 1e-8:
src/tesis_v2/metricas.py:61: RuntimeWarning: All-NaN slice encountered
  if np.nanmax(np.abs(np.r_[da, db])) < 1e-8:


Réplicas con pendiente aparente no definida: 72 de 500


,metrica,aparente,optimismo_medio,corregida,validacion_cruzada_5x20
0,log_loss,0.1298,-0.0059,0.1358,0.1387
1,auc,0.7260,0.0514,0.6746,0.6294
2,pendiente_cal,1.3121,NaN,1.4452,0.6181
3,brier_escalado,0.0280,0.0163,0.0117,0.0046


**Lectura.** El *bootstrap* de Harrell estima un optimismo de unos 0,05 en AUC y corrige el AUC aparente (0,73) a cerca de 0,67. La pendiente corregida es la mediana de la pendiente de cada modelo *bootstrap* evaluado en la muestra original.

**Pendiente.** Las dos vías se contradicen:

- la validación cruzada da 0,62 (predicciones algo extremas);
- Harrell da 1,42 (predicciones demasiado tímidas, porque en muchas réplicas el pliegue interno elige la penalización máxima).

Junto con el IC de la validación cruzada (−0,42 a 1,65), esto dice algo importante: **con 23 eventos la pendiente de calibración no se puede estimar con precisión**. La regla la usa como filtro grueso, para descartar sobreajustes evidentes como 0,08 o 0,22, y no como una medida fina. Es más optimista que la validación cruzada repetida (0,63), algo conocido en muestras pequeñas con modelos muy penalizados: el *bootstrap* repite un 63 % de los clientes y subestima la variabilidad de la selección de hiperparámetros. Por eso la validación cruzada 5 × 20 sigue siendo la estimación principal, y Harrell una cota superior razonable.

## 6. Beneficio neto (análisis de curva de decisión)

El beneficio neto responde una pregunta práctica: si rechazo a todos los solicitantes con PD por encima de un umbral t, ¿gano algo frente a aprobar a todos o a rechazar a todos? El umbral expresa cuántos buenos estoy dispuesto a rechazar por cada malo evitado: t/(1 − t). En libranza, con márgenes de crédito de consumo, el rango relevante está entre 2 % y 15 %.

In [8]:
umbrales = np.arange(0.01, 0.151, 0.005)
fig, ax = plt.subplots(figsize=(7.5, 4))
for k, col in zip(["M2", "M3", "M5", "M6"], figuras.SECUENCIA):
    nb = np.mean([mt.beneficio_neto(y, p, umbrales)["modelo"] for p in Ps[k]], axis=0)
    ax.plot(umbrales, 1000 * nb, color=col, lw=1.7, label=k)
todos = mt.beneficio_neto(y, Ps["M2"][0], umbrales)["marcar_todos"]
ax.plot(umbrales, 1000 * todos, color=figuras.GRIS, ls="--", lw=1, label="Rechazar a todos")
ax.axhline(0, color="black", lw=0.8, label="Aprobar a todos")
ax.set_ylim(-5, 12)
ax.set_xlabel("Umbral de PD para rechazar")
ax.set_ylabel("Beneficio neto (por 1.000 solicitudes)")
ax.set_title("Curva de decisión (predicciones fuera de pliegue, media de 20 repeticiones)")
ax.legend(fontsize=8)
figuras.guardar(fig, "fase4", "M03", "curva_decision",
                "Solo entre umbrales de 3 % y 6 % los modelos superan a aprobar o rechazar a todos (hasta ≈ 8 por 1.000); por encima de 7 % no aportan.")

## 7. Qué podemos y qué no podemos afirmar

Esta sección la exige el plan y la escribo con los números de la potencia de la Fase 1. Con 23 eventos, el IC del AUC tiene un ancho de unos ±0,12 (lo confirma la tabla: M2 va de 0,50 a 0,75) y la diferencia mínima detectable entre dos modelos es de unos 0,18.

**Podemos afirmar:**

- Con datos propios hay una **señal de discriminación modesta pero real**. M2, M3, M4 y M5 tienen un AUC fuera de pliegue de 0,63 a 0,65, con límite inferior ≥ 0,50.
- La señal solo se aprovecha con **contracción fuerte**:
  - el pliegue interno elige casi siempre la penalización más fuerte de M3 (τ = 0,25 en 97 de 100 ajustes) y C ≤ 0,1 en M2;
  - los modelos sin contracción adicional (*scorecard*, Firth) sobreajustan, con pendientes de 0,08 y 0,22 y un *log loss* peor que la tasa base.
- Los modelos contraídos (M2, M3, M5, M4, M7) **superan al scorecard M1** en *log loss*, con IC que excluyen el 0 y p de Holm ≤ 0,02.
- La configuración v1 (apilado con Home Credit al 80 %) tiene una pendiente de 0,57, fuera del rango aceptable. Usar Home Credit como *prior* con escala ajustable la mejora en calibración y en *log loss*.
- La **regla de políticas evaluable señala al grupo equivocado** (AUC 0,39).

**No podemos afirmar:**

- Que algún modelo estime la probabilidad mejor que la tasa base. Es el paso 4 de la regla y lo sostengo tal cual en la tesis.
- Que un modelo discrimine mejor que otro. Todas las diferencias de AUC entre M2, M3, M4, M5 y M7 están dentro del ruido (|Δ| ≤ 0,03).
- Que el *prior* de Home Credit mejore de forma significativa a la logística penalizada. La diferencia va a su favor, pero el IC pareado incluye 0 (p de Holm = 1).
- Que las predicciones estén calibradas por encima del 8 % de PD: hay muy pocos eventos en esa zona.

In [9]:
_ = io_datos.escribir_registro("10_comparacion_y_seleccion", {
    "bootstrap": 2000, "elegido": elegido, "mejor_log_loss": mejor, "mejora_frente_tasa_base_demostrable": bool(demuestra),
    "harrell_replicas": 500})
figuras.escribir_catalogo("fase4", "Figuras de la Fase 4")

PosixPath('resultados/fase4/catalogo_figuras.md')